In [ ]:
import os
import time

import bibtexparser
import flatdict as fd
import numpy as np
import pandas as pd

from pynxtools_em.examples.oasisb_bibliography import get_bibliographical_metadata
from pynxtools_em.examples.oasisb_openalex import get_data_for_doi_from_openalex
from pynxtools_em.examples.oasisb_utils import get_project_id

rng = np.random.default_rng(seed=42)

print(os.getcwd())
with open("source_directory.txt") as fp:
    src_directory = f"{fp.readline().strip().replace('/', os.sep)}"
print(src_directory)

In [ ]:
spread_sheet_of_all_projects = pd.read_excel(
    f"{src_directory}{os.sep}aaa_legacy_data.ods",
    sheet_name="aaa_legacy_data",
    engine="odf",
)
with open(f"{src_directory}{os.sep}aaa_legacy_data.bib") as fp:
    bib = bibtexparser.load(fp).entries_dict

In [ ]:
api_queries_cnt = 0
api_queries_max = 10
for row in spread_sheet_of_all_projects.itertuples(index=True):
    if row.parse in (1, 2):
        if int(row.project_name) < 0:  # skip all before
            continue

        # print(row.project_name)
        project_id = get_project_id(f"{row.project_name}")
        data_and_paper = get_bibliographical_metadata(bib, project_id)

        n_queries = get_data_for_doi_from_openalex(bib, data_and_paper)

        api_queries_cnt += n_queries  # sleep only when necessary
        if api_queries_cnt >= api_queries_max:
            sleep = float(rng.uniform(1, 10))
            print(f"Sleeping for {sleep}s")
            time.sleep(sleep)
            api_queries_cnt = 0
print("Batch querying queue done")

In [ ]:
# HTTPError: 404 Client Error: Not Found for url: https://api.openalex.org/works/https://doi.org/10.18150/SRK93E, 249
# HTTPError: 404 Client Error: Not Found for url: https://api.openalex.org/works/https://doi.org/10.3030/956099, 434
# HTTPError: 404 Client Error: Not Found for url: https://api.openalex.org/works/https://doi.org/10.5281/zenodo.13270697, 460
# HTTPError: 404 Client Error: Not Found for url: https://api.openalex.org/works/https://doi.org/10.5281/zenodo.14854284, 481
# HTTPError: 404 Client Error: Not Found for url: https://api.openalex.org/works/https://doi.org/10.5281/zenodo.5815072, 504
# HTTPError: 404 Client Error: Not Found for url: https://api.openalex.org/works/https://doi.org/10.3030/101035013, 506
# HTTPError: 404 Client Error: Not Found for url: https://api.openalex.org/works/https://doi.org/10.5281/zenodo.10775819, 518
# HTTPError: 404 Client Error: Not Found for url: https://api.openalex.org/works/https://doi.org/10.18150/WUEUCR, 570
# HTTPError: 404 Client Error: Not Found for url: https://api.openalex.org/works/https://doi.org/10.5281/ZENODO.14524981, 579

In [ ]:
json_files = set()
for root, dirs, files in os.walk("openalex"):
    for file in files:
        # fpath = f"{root}/{file}".replace(os.sep * 2, os.sep)
        json_files.add(file)
print(len(json_files))

In [ ]:
mdata = fd.FlatDict(data, delimiter="/")
for key, obj in mdata.items():
    print(f"{key}, {obj}")

In [ ]:
authorships = data.get("authorships", [])
if len(authorships) == 0:
    print("No authorships")

first_author = authorships[0]

institutions = first_author.get("institutions", [])
if not institutions:
    print("No institutions")

first_institution = institutions[0]

result = {
    "doi": doi,
    "first_author": first_author.get("author", {}).get("display_name"),
    "institution": first_institution.get("display_name"),
    "country_code_iso3": first_institution.get("country_code"),
    "town": first_institution.get("city"),
}
print(result)